# Day 57: Anomaly Detection Basics

## The Problem
Many high-value data science problems — fraud detection, manufacturing defects, sensor/equipment failure, intrusion detection — share a pattern: the interesting cases are rare and look different from the bulk of "normal" data. Because anomalies are rare, you usually can't train an ordinary supervised classifier on them (too few positive examples, and new kinds of anomalies won't resemble past ones), so anomaly detection instead tries to learn what "normal" looks like and flag anything that deviates significantly from it. This cell builds a toy 2D dataset to demonstrate the idea: 200 points drawn from a standard normal distribution (clustered near the origin, representing "normal" behavior) plus 10 points drawn from a much wider uniform range (representing rare, unusual observations). We'll use this same `X` to compare two different anomaly detection algorithms below.

In [ ]:
import numpy as np

np.random.seed(0)
# "Normal" data: 200 points clustered around the origin (standard normal distribution)
normal = np.random.randn(200, 2)
# "Anomalies": 10 points spread widely and uniformly, likely to fall far from the normal cluster
anomalies = np.random.uniform(low=-6, high=6, size=(10, 2))
# Combine into one dataset the way real data would arrive — anomalies mixed in, unlabeled
X = np.vstack([normal, anomalies])

## Isolation Forest
Isolation Forest takes a clever, different approach from most anomaly detectors: instead of modeling what "normal" looks like directly, it builds an ensemble of random trees that repeatedly pick a random feature and a random split value to partition the data. The intuition is that anomalies — points that are few and different — get separated ("isolated") from the rest of the data in just a few random splits, while normal points, sitting in dense regions, take many more splits to isolate because they're surrounded by similar points. The average path length (number of splits) to isolate a point across all trees becomes its anomaly score; points with unusually short average paths are flagged as anomalies. The `contamination=0.05` parameter tells the model to expect roughly 5% of the data to be anomalous, which it uses to set the threshold on that score — note this should roughly match your real prior belief about how much of your data is anomalous, since it directly determines how many points get flagged.

In [ ]:
from sklearn.ensemble import IsolationForest

# contamination=0.05 tells the model to expect ~5% of points to be anomalies,
# which sets the score threshold used to flag them
iso = IsolationForest(contamination=0.05, random_state=42)
preds = iso.fit_predict(X)  # returns 1 for normal points, -1 for flagged anomalies
print('Number flagged as anomalies:', (preds == -1).sum())

## One-Class SVM
One-Class SVM takes yet another approach: it tries to learn a boundary (in a possibly transformed, higher-dimensional feature space, via the kernel trick) that tightly encloses the region where most of the "normal" training data lives, then flags any point falling outside that learned boundary as an anomaly. The `kernel='rbf'` setting lets the boundary be a flexible, non-linear shape rather than a straight line/plane, which matters because normal data rarely forms a simple convex region. The `nu` parameter (here 0.05) is roughly an upper bound on the fraction of training points allowed to be treated as outliers (and a lower bound on the fraction of points that can end up as "support vectors" defining the boundary) — conceptually similar to `contamination` in Isolation Forest, though the underlying math is different. Compare its flagged count to Isolation Forest's below to see how differently these algorithms can behave on the same data.

In [ ]:
from sklearn.svm import OneClassSVM

# nu controls the expected fraction of outliers (similar role to `contamination` above);
# kernel='rbf' allows a flexible, non-linear decision boundary around "normal" data
ocsvm = OneClassSVM(nu=0.05, kernel='rbf', gamma='scale')
preds_svm = ocsvm.fit_predict(X)  # 1 = normal, -1 = flagged anomaly
print('Number flagged as anomalies:', (preds_svm == -1).sum())

## Visualizing Anomalies
Numbers alone (like "12 anomalies flagged") don't tell you whether a model is actually finding the right points, so it's good practice to plot the data and color by the model's predictions. Here, points the Isolation Forest labeled `-1` (anomalous) are colored red and everything else blue. In a well-behaved case like this synthetic example, you'd expect most red points to sit away from the dense central blob — if instead red points are scattered randomly throughout the normal cluster, that's a sign the model's `contamination` setting or its assumptions don't fit this data well.

In [ ]:
import matplotlib.pyplot as plt

# Color points by the Isolation Forest's prediction: red = flagged anomaly, blue = normal
colors = ['red' if p == -1 else 'blue' for p in preds]
plt.scatter(X[:, 0], X[:, 1], c=colors, alpha=0.6)
plt.title('Isolation Forest: Anomalies in Red')
plt.show()